# 01 — Raw folder to Stage 1 HDF5

This notebook is the ingestion entry point for a new dataset. It discovers raw files, joins patient labels, previews the actual parser output, builds a validated manifest, and writes one or more nested cell-count levels to `stage1_raw_data.h5`.

## What is read

- **FCS (`.fcs`)**: read with FlowIO. Events become `[cells, channels]`; marker names use FCS `PnS` and fall back to `PnN`. Install with `pip install -e '.[fcs,notebook]'`. Compensation and biological transforms are intentionally **not** guessed here; apply them explicitly during preprocessing.
- **CSV/TSV/TXT**: rows are cells and columns are markers. A text header supplies marker names. Headerless numeric files need `markers_by_tube`.
- **NPY**: a 2-D numeric `[cells, markers]` array. Supply `markers_by_tube` for meaningful names.
- **NPZ**: use key `cells` (or the first array); an optional `markers` array supplies names.

Labels are never inferred from FCS metadata. They are joined from an explicit sample-information table and checked for consistency across tubes. `counts` records the original number of events before subsampling.

## Expected input layout

Point `INPUT_DIR` at the folder containing the dataset folders and `OUTPUT_DIR` at any separate writable results folder. The presets expect:

```text
INPUT_DIR/
├── BLAST110/{FCS/, labels/, sample_info.csv}
├── LAIP29/{FCS/, labels/, annotations/, sample_info.csv}
└── FlowCAPII/{FCS/, attachments/AML.csv}
OUTPUT_DIR/
├── manifests/                     # generated here
└── data/stage1_raw_data.h5        # generated here
```

BLAST110 and LAIP29 are the public cMRD layouts. FlowCAPII is the third dataset used by the legacy FlowCode archive.

In [ ]:
from collections import Counter
from pathlib import Path
import re

import h5py
import numpy as np
import pandas as pd

from flowlot.io import (
    audit_manifest,
    audit_stage1,
    build_stage1_from_manifest,
    create_manifest_from_folder,
    create_manifest_from_metadata,
    load_cytometry_file,
)
from flowlot.io.manifest import SUPPORTED_INPUTS, is_ignored_input

## 1. Select the dataset and edit the two directories

Set `INPUT_DIR`, `OUTPUT_DIR`, and `DATASET_NO`. Dataset 1 and 2 follow the cMRD download structure. Dataset 3 follows the legacy FlowCAPII structure and obtains patient/tube/class fields from `attachments/AML.csv`. Only the selected preset continues through later cells.

In [ ]:
# -------------------- USER INPUTS --------------------
INPUT_DIR = Path('/standard/vol194/g_bme-RohdeLab/FlowLOT_main/Data/Raw_Data/').expanduser().resolve()
OUTPUT_DIR = Path('/standard/vol194/g_bme-RohdeLab/Naqib/2026_AML_projects_folder/Projects/P1_FlowLOT_v1/Processed_data/').expanduser().resolve()
DATASET_NO = 2  # 1=BLAST110, 2=LAIP29, 3=FlowCAPII
# -----------------------------------------------------

DATASET_PRESETS = {
    1: {
        'dataset': 'BLAST110',
        'manifest_mode': 'folder',
        'raw_root': INPUT_DIR / 'BLAST110/FCS',
        'labels_csv': INPUT_DIR / 'BLAST110/sample_info.csv',
        'manifest': OUTPUT_DIR / 'manifests/blast110.csv',
        'filename_pattern': r'BLAST110_(?P<patient_id>[0-9]+)_(?P<tube_id>P[0-9]+)\.(?:fcs|csv|tsv|txt|npy|npz)$',
        # Legacy sample_info.csv joins by full filename stem, not patient_id.
        'labels_id_column': 'BLAST110_ID',
        'labels_label_column': 'sample_type',
        'labels_match': 'file_id',
        # One event-level CSV per raw file: event_ID,WBC,Blast[,LAIP].
        'event_labels_root': INPUT_DIR / 'BLAST110/labels',
        'event_labels_pattern': '{stem}.csv',
        'event_id_column': 'event_ID',
        'population_columns': ['WBC', 'Blast'],
        # Needed for NPY/headerless text; FCS/headered CSV usually infer names.
        'markers_by_tube': {
            # 'T1': ['FSC-A', 'SSC-A', 'CD45', 'CD34'],
        },
        'cell_counts': [500, 1000, 2000],  # or ['all']
    },
    2: {
        'dataset': 'LAIP29',
        'manifest_mode': 'folder',
        'raw_root': INPUT_DIR / 'LAIP29/FCS',
        'labels_csv': INPUT_DIR / 'LAIP29/sample_info.csv',
        'manifest': OUTPUT_DIR / 'manifests/laip29.csv',
        'filename_pattern': r'LAIP29_(?P<patient_id>[0-9]+_(?:Dx|FU))_(?P<tube_id>P[0-9]+)\.(?:fcs|csv|tsv|txt|npy|npz)$',
        'labels_id_column': 'LAIP29_ID',
        'labels_label_column': 'sample_type',
        'labels_match': 'file_id',
        'event_labels_root': INPUT_DIR / 'LAIP29/labels',
        'event_labels_pattern': '{stem}.csv',
        'event_id_column': 'event_ID',
        'population_columns': ['WBC', 'Blast', 'LAIP'],
        'markers_by_tube': {},
        'cell_counts': [500, 1000, 2000],
    },
    3: {
        'dataset': 'FLOWCAPII',
        'manifest_mode': 'metadata',
        'raw_root': INPUT_DIR / 'FlowCAPII/FCS',
        'metadata_csv': INPUT_DIR / 'FlowCAPII/attachments/AML.csv',
        'manifest': OUTPUT_DIR / 'manifests/flowcapii.csv',
        'file_column': 'FCS file',
        'patient_id_column': 'Individual',
        'tube_id_column': 'Tube number',
        'label_column': 'Condition',
        'tube_prefix': 'P',
        'markers_by_tube': {},
        'cell_counts': [500, 1000, 2000],
    },
}
if DATASET_NO == 1:
    selected_dataset = DATASET_PRESETS[1]
elif DATASET_NO == 2:
    selected_dataset = DATASET_PRESETS[2]
elif DATASET_NO == 3:
    selected_dataset = DATASET_PRESETS[3]
else:
    raise ValueError(f'DATASET_NO must be 1, 2, or 3; received {DATASET_NO}')
DATASETS = [selected_dataset]
print(f"Selected dataset {DATASET_NO}: {DATASETS[0]['dataset']}")
print('Input directory:', INPUT_DIR)
print('Output directory:', OUTPUT_DIR)

STAGE1 = OUTPUT_DIR / 'data/stage1_raw_data.h5'
SEED = 42
RECURSIVE = True
STRICT_FILENAMES = True
GENERATE_MANIFESTS = False  # inspect discovery first, then set True once
OVERWRITE_MANIFESTS = False
RUN_BUILD = False           # audit/preview first, then set True

BLAST110 and LAIP29 have two label inputs: `sample_info.csv` supplies the sample class, and one CSV per FCS file supplies event-level populations. FlowCAPII instead uses `attachments/AML.csv` for the raw filename, individual, tube, and condition and has no event-label sidecar in the legacy pipeline.

In [ ]:
display(pd.DataFrame({'BLAST110_ID': ['BLAST110_1_P1'], 'sample_type': ['AML_Dx']}))
display(pd.DataFrame({'event_ID': [1, 2, 3], 'WBC': [1, 1, 0], 'Blast': [1, 0, 0]}))

## 2. Discover files before writing anything

This dry run shows which folder is read, which files are eligible, and which names fail the patient/tube rule.

In [ ]:
for cfg in DATASETS:
    root = cfg['raw_root'].resolve()
    if not root.is_dir():
        print(f"MISSING raw_root: {root}")
        continue
    iterator = root.rglob('*') if RECURSIVE else root.glob('*')
    files = sorted(
        p for p in iterator
        if p.is_file() and p.suffix.lower() in SUPPORTED_INPUTS
        and not is_ignored_input(p, root)
    )
    print(f"\n{cfg['dataset']}: reading {root}")
    print(f"supported={len(files)}, formats={dict(Counter(p.suffix.lower() for p in files))}")
    if cfg['manifest_mode'] == 'folder':
        pattern = re.compile(cfg['filename_pattern'])
        matched = [p for p in files if pattern.search(p.relative_to(root).as_posix())]
        unmatched = [p.relative_to(root).as_posix() for p in files if p not in matched]
        print(f'matched={len(matched)}')
        print('matched examples:', [p.relative_to(root).as_posix() for p in matched[:8]])
        if unmatched:
            print('UNMATCHED examples:', unmatched[:8])
    else:
        metadata_path = cfg['metadata_csv']
        print('metadata table:', metadata_path)
        if metadata_path.exists():
            metadata_preview = pd.read_csv(metadata_path)
            print('metadata columns:', metadata_preview.columns.tolist())
            display(metadata_preview.head())
        else:
            print('MISSING metadata table')

## 3. Generate the manifest

Set `GENERATE_MANIFESTS=True` only after discovery looks correct. Paths are stored relative to the manifest, making the project movable. Generation refuses to overwrite an existing manifest unless explicitly enabled.

In [ ]:
if GENERATE_MANIFESTS:
    for cfg in DATASETS:
        if cfg['manifest_mode'] == 'folder':
            frame = create_manifest_from_folder(
                raw_root=cfg['raw_root'], output=cfg['manifest'],
                filename_pattern=cfg['filename_pattern'], labels=cfg['labels_csv'],
                markers_by_tube=cfg.get('markers_by_tube'),
                labels_id_column=cfg['labels_id_column'],
                labels_label_column=cfg['labels_label_column'],
                labels_match=cfg['labels_match'],
                event_labels_root=cfg['event_labels_root'],
                event_labels_pattern=cfg['event_labels_pattern'],
                event_id_column=cfg['event_id_column'],
                population_columns=cfg['population_columns'],
                recursive=RECURSIVE, strict=STRICT_FILENAMES,
                overwrite=OVERWRITE_MANIFESTS,
            )
        else:
            frame = create_manifest_from_metadata(
                raw_root=cfg['raw_root'], metadata=cfg['metadata_csv'],
                output=cfg['manifest'], file_column=cfg['file_column'],
                patient_id_column=cfg['patient_id_column'],
                tube_id_column=cfg['tube_id_column'], label_column=cfg['label_column'],
                tube_prefix=cfg['tube_prefix'], markers_by_tube=cfg['markers_by_tube'],
                overwrite=OVERWRITE_MANIFESTS,
            )
        print(f"wrote {cfg['manifest']} ({len(frame)} patient/tube rows)")
else:
    print('Dry run only: set GENERATE_MANIFESTS=True after reviewing discovery.')

## 4. Audit labels, tubes, paths, and marker declarations

The audit catches missing files, duplicate patient/tube rows, conflicting patient labels, non-finite values in the manifest, and invalid marker declarations.

In [ ]:
for cfg in DATASETS:
    if not cfg['manifest'].exists():
        print(f"MISSING manifest: {cfg['manifest']}")
        continue
    inventory, issues = audit_manifest(cfg['manifest'])
    print(f"\n{cfg['dataset']}: rows={len(inventory)}, issues={len(issues)}")
    display(inventory.head())
    if not issues.empty:
        display(issues)

## 5. Parse a few real files before building HDF5

This invokes the same raw reader as the builder and verifies matrix shape, actual PnS/PnN marker names, event-level CSV columns, and original WBC/Blast/LAIP sums. It stops on unresolved `channel_N` names rather than silently losing marker identity. During the build, rows are joined by `event_ID` before subsampling. Increase `PREVIEW_FILES` cautiously for very large FCS files.

In [ ]:
PREVIEW_FILES = 3
preview_rows = []
for cfg in DATASETS:
    manifest_path = cfg['manifest'].resolve()
    if not manifest_path.exists():
        continue
    manifest = pd.read_csv(manifest_path, dtype=str).fillna('')
    for row in manifest.head(PREVIEW_FILES).itertuples(index=False):
        source = Path(row.path)
        if not source.is_absolute():
            source = manifest_path.parent / source
        declared = [m.strip() for m in row.markers.split(';') if m.strip()] or None
        try:
            cells, markers = load_cytometry_file(source, declared)
            population_summary = {}
            event_labels_value = getattr(row, 'event_labels_path', '')
            if event_labels_value:
                event_labels_path = Path(event_labels_value)
                if not event_labels_path.is_absolute():
                    event_labels_path = manifest_path.parent / event_labels_path
                event_labels = pd.read_csv(event_labels_path)
                populations = [p for p in getattr(row, 'population_columns', '').split(';') if p]
                population_summary = event_labels[populations].sum().to_dict()
            preview_rows.append({
                'dataset': cfg['dataset'], 'patient_id': row.patient_id, 'tube_id': row.tube_id,
                'file': source.name, 'shape': cells.shape, 'original_count': len(cells),
                'finite_fraction': float(np.isfinite(cells).mean()),
                'markers': markers[:8], 'original_population_counts': population_summary,
                'status': 'OK',
            })
        except Exception as exc:
            preview_rows.append({'dataset': cfg['dataset'], 'file': source.name, 'status': repr(exc)})
display(pd.DataFrame(preview_rows))

## 6. Expand and build every requested cell count

`cell_counts: [500, 1000, 2000]` creates three HDF5 levels. With the same seed, smaller levels are nested subsets of larger levels (`500 ⊂ 1000 ⊂ 2000`) whenever enough cells exist. The first write creates the file and later writes append new levels. Existing target groups are protected unless `overwrite=True` is passed deliberately.

In [ ]:
CONFIGURATIONS = [
    {'dataset': cfg['dataset'], 'manifest': cfg['manifest'], 'cells': cells}
    for cfg in DATASETS
    for cells in cfg['cell_counts']
]
pd.DataFrame(CONFIGURATIONS)

In [ ]:
if RUN_BUILD:
    for index, cfg in enumerate(CONFIGURATIONS):
        build_stage1_from_manifest(
            manifest=cfg['manifest'],
            output=STAGE1,
            dataset_name=cfg['dataset'],
            subsampled_cell_count=cfg['cells'],
            seed=SEED,
            mode='w' if index == 0 else 'a',
        )
        print(f"built {cfg['dataset']}/{cfg['cells']}")
else:
    print('No HDF5 written. Set RUN_BUILD=True after all previews and audits pass.')

## 7. Verify the Stage 1 file and summarize every split

This checks the stored schema and reports patient/tube coverage, event counts, and the exact sampled/original WBC, Blast, and LAIP values. A stored count may be below the requested level when fewer annotated events are available.

In [ ]:
if STAGE1.exists():
    stage1_inventory, stage1_issues = audit_stage1(STAGE1)
    print(f'Stage 1 rows={len(stage1_inventory)}, issues={len(stage1_issues)}')
    if not stage1_issues.empty:
        display(stage1_issues)
    statistics = []
    population_statistics = []
    with h5py.File(STAGE1, 'r') as handle:
        for dataset_name in handle:
            for cell_level in handle[dataset_name]:
                for sample_key, sample in handle[dataset_name][cell_level].items():
                    for tube_id, tube in sample.items():
                        matrix = tube['raw_cell_matrix']
                        statistics.append({
                            'dataset': dataset_name, 'cell_level': cell_level,
                            'patient_id': tube.attrs.get('patient_id', sample_key),
                            'label': tube.attrs.get('label'), 'tube_id': tube_id,
                            'stored_cells': matrix.shape[0], 'markers': matrix.shape[1],
                            'original_count': int(tube.attrs['counts']),
                        })
                        if 'population_counts' in tube:
                            counts = tube['population_counts']
                            names = [value.decode() for value in counts.attrs['population_names']]
                            metrics = [value.decode() for value in counts.attrs['metric_names']]
                            for pop_index, population in enumerate(names):
                                population_statistics.append({
                                    'dataset': dataset_name, 'cell_level': cell_level,
                                    'patient_id': tube.attrs.get('patient_id', sample_key),
                                    'tube_id': tube_id, 'population': population,
                                    **dict(zip(metrics, counts[pop_index])),
                                })
    stats = pd.DataFrame(statistics)
    display(stats.head())
    display(stats.groupby(['dataset', 'cell_level', 'tube_id']).agg(
        patients=('patient_id', 'nunique'),
        labels=('label', 'nunique'),
        stored_min=('stored_cells', 'min'),
        stored_median=('stored_cells', 'median'),
        original_min=('original_count', 'min'),
        original_max=('original_count', 'max'),
    ).reset_index())
    if population_statistics:
        display(pd.DataFrame(population_statistics))
else:
    print(f"No Stage 1 file yet: {STAGE1.resolve()}")

## Troubleshooting

- **FCS import error**: install the `fcs` extra and restart the kernel.
- **Unmatched files**: inspect relative paths in the discovery cell and adjust `filename_pattern`; do not silently relabel them.
- **Missing labels**: ensure IDs in the label CSV exactly match IDs captured from filenames (including leading zeros).
- **Marker-count mismatch**: the declared marker list must have one entry per matrix column.
- **Mixed panels**: map marker names separately under each `tube_id`.
- **Need to rebuild**: delete/move the output deliberately or use a new output name; safeguards prevent accidental replacement.